# Phase 1 — Descarga NASA FIRMS (VIIRS active fire)

**Entorno**: Local  
**Tiempo estimado**: ~2 min  
**Output**: CSV consolidado en `data/firms/viirs_snpp_corrientes_2021-12_2022-02.csv`

Descarga detecciones de fuego activo VIIRS (Suomi NPP Standard Product) sobre Corrientes,
Argentina para dic 2021 – feb 2022. VIIRS_SNPP_SP (datos históricos) tiene límite de 1 día
por petición — se hacen 90 peticiones individuales.

## 1. Imports y credenciales

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

FIRMS_KEY = os.getenv("FIRMS_API_KEY")

DATA_DIR = Path("..") / "data" / "firms"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"FIRMS key: {FIRMS_KEY[:8]}...")
print(f"Output dir: {DATA_DIR.resolve()}")

## 2. Parámetros de descarga

In [ ]:
# Corrientes, Argentina — mismo BBOX que Sentinel-2
BBOX_W, BBOX_S, BBOX_E, BBOX_N = -59.5, -29.0, -56.0, -26.5

# Período completo
START_DATE = datetime(2021, 12, 1)
END_DATE   = datetime(2022, 2, 28)

# Producto: VIIRS Suomi NPP Standard Product (histórico)
# NOTA: VIIRS_SNPP_SP tiene límite de days=1 por petición (no 10 como NRT)
PRODUCT = "VIIRS_SNPP_SP"
CHUNK_DAYS = 1

# Una petición por día
chunks = []
current = START_DATE
while current <= END_DATE:
    chunks.append((current, 1))
    current += timedelta(days=1)

print(f"Período: {START_DATE.date()} → {END_DATE.date()}")
print(f"Total peticiones: {len(chunks)} (1 día cada una)")
print(f"Tiempo estimado: ~{len(chunks) * 0.5 / 60:.1f} min")

## 3. Descarga por chunks

In [ ]:
def download_firms_day(key, product, bbox_w, bbox_s, bbox_e, bbox_n, date_str):
    bbox_str = f"{bbox_w},{bbox_s},{bbox_e},{bbox_n}"
    url = (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{key}/{product}/{bbox_str}/1/{date_str}"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text


all_dfs = []
errors = []

for i, (start, _) in enumerate(chunks):
    date_str = start.strftime("%Y-%m-%d")
    output_path = DATA_DIR / f"viirs_{date_str}.csv"

    if output_path.exists():
        df = pd.read_csv(output_path)
        if len(df) > 0:
            all_dfs.append(df)
        continue

    try:
        csv_text = download_firms_day(
            FIRMS_KEY, PRODUCT, BBOX_W, BBOX_S, BBOX_E, BBOX_N, date_str
        )

        if csv_text.startswith("Error") or csv_text.startswith("<!DOCTYPE"):
            raise ValueError(f"API error: {csv_text[:100]}")

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(csv_text)

        lines = csv_text.strip().split("\n")
        n_det = len(lines) - 1
        if n_det > 0:
            from io import StringIO
            df = pd.read_csv(StringIO(csv_text))
            all_dfs.append(df)
            if i % 10 == 0 or n_det > 50:
                print(f"  {date_str}  {n_det:>5} detecciones")

        time.sleep(0.5)

    except Exception as e:
        print(f"  ERROR {date_str}: {e}")
        errors.append((date_str, str(e)))

print(f"\nDías procesados: {len(chunks) - len(errors)}/{len(chunks)}")
print(f"Errores: {len(errors)}")
if errors:
    print(errors[:3])

## 4. Consolidar en un único CSV

In [ ]:
if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    df_all["acq_date"] = pd.to_datetime(df_all["acq_date"])
    df_all = df_all.sort_values("acq_date").reset_index(drop=True)

    output_path = DATA_DIR / "viirs_snpp_corrientes_2021-12_2022-02.csv"
    df_all.to_csv(output_path, index=False)

    print(f"CSV consolidado: {output_path}")
    print(f"Total detecciones: {len(df_all)}")
    print(f"Columnas: {list(df_all.columns)}")
    print(f"\nPrimeras filas:")
    print(df_all.head())
else:
    print("Sin datos para consolidar.")

## 5. Visualización — detecciones por día

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if 'df_all' in dir() and len(df_all) > 0:
    daily = df_all.groupby("acq_date").size().rename("detections")

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(daily.index, daily.values, alpha=0.6, color="#d62728")
    ax.plot(daily.index, daily.values, color="#d62728", linewidth=1)

    ax.axvspan(
        pd.Timestamp("2021-12-01"), pd.Timestamp("2021-12-31"),
        alpha=0.08, color="gray", label="pre-fire"
    )
    ax.axvspan(
        pd.Timestamp("2022-01-01"), pd.Timestamp("2022-01-31"),
        alpha=0.08, color="red", label="peak fire"
    )
    ax.axvspan(
        pd.Timestamp("2022-02-01"), pd.Timestamp("2022-02-28"),
        alpha=0.08, color="green", label="post-fire"
    )

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.xticks(rotation=45)
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Detecciones VIIRS")
    ax.set_title("Detecciones de fuego activo — Corrientes, Argentina (VIIRS SNPP)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    out = Path("..") / "results" / "firms_daily_detections.png"
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"Gráfico guardado en {out}")

---
**OK → continuar con `03_download_era5.ipynb`**